# Experiment report template

把这个 notebook 复制到任意 `mix/output/[ID]-.../` 目录下运行。它会自动从当前目录名解析实验 ID，并调用 `analysis/deep_analyse.py` 里的 `get_basic_result()` 和 `get_analyser()`。

如果目录名里没有 `[ID]`，手动设置下面 setup cell 里的 `EXP_ID`。

In [ ]:
from pathlib import Path
import re
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 80)
plt.rcParams['figure.dpi'] = 120

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'analysis' / 'deep_analyse.py').exists():
            return p
    raise FileNotFoundError('Cannot find repo root containing analysis/deep_analyse.py')

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = find_repo_root(NOTEBOOK_DIR)
ANALYSIS_DIR = REPO_ROOT / 'analysis'
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

# The repository file is named deep_analyse.py.
from deep_analyse import get_basic_result, get_analyser, show_fct

m = re.search(r'\[(\d+)\]', NOTEBOOK_DIR.name)
EXP_ID = m.group(1) if m else None
# EXP_ID = '123'  # Uncomment and edit if this notebook is not under mix/output/[ID]-...
if EXP_ID is None:
    raise ValueError('Could not infer EXP_ID from current directory name. Set EXP_ID manually in this cell.')

ana = get_analyser(EXP_ID)
print('repo root:', REPO_ROOT)
print('experiment id:', EXP_ID)
print('experiment dir:', ana.dir)

## Basic result

`get_basic_result()` 给出最核心的 normalized FCT 总览：总体、intra、inter 的平均值和 P99。

In [ ]:
basic = get_basic_result(EXP_ID)
display(basic)

avg_all, avg_intra, avg_inter = ana.get_avg_fct()
p99_all, p99_intra, p99_inter = ana.get_p99_fct()
large_avg, large_inter_avg, large_inter_p99, large_intra_avg = ana.get_large_flow_fct()
small_avg, small_inter_avg, small_inter_p99, small_intra_avg = ana.get_small_flow_fct()

detail = pd.DataFrame([
    {'metric': 'Avg FCT slowdown', 'all': avg_all, 'intra': avg_intra, 'inter': avg_inter},
    {'metric': 'P99 FCT slowdown', 'all': p99_all, 'intra': p99_intra, 'inter': p99_inter},
    {'metric': 'Large flow avg slowdown', 'all': large_avg, 'intra': large_intra_avg, 'inter': large_inter_avg},
    {'metric': 'Large flow inter P99 slowdown', 'all': np.nan, 'intra': np.nan, 'inter': large_inter_p99},
    {'metric': 'Small flow avg slowdown', 'all': small_avg, 'intra': small_intra_avg, 'inter': small_inter_avg},
    {'metric': 'Small flow inter P99 slowdown', 'all': np.nan, 'intra': np.nan, 'inter': small_inter_p99},
])
display(detail)

## Config and available logs

先看这次实验的关键配置和当前目录里有哪些日志文件。

In [ ]:
interesting_keys = [
    'MSG', 'CC_MODE', 'WAN_CC_MODE', 'TOPOLOGY_FILE', 'FLOW_FILE', 'TCP_FLOW_FILE',
    'ENABLE_W', 'INV_DELTA', 'W_MAX', 'W_K', 'BETA', 'WAN_EPOCH_US', 'GSCC_FAIR',
    'BUFFER_SIZE', 'WAN_BUFFER_SIZE', 'DCI_BUFFER_SIZE',
    'UNO_PHANTOM_ENABLED', 'UNO_BDP_BYTES', 'UNO_INTRA_RTT_NS',
]
config_rows = []
for k in interesting_keys:
    if k in ana.config:
        config_rows.append({'key': k, 'value': ana.config[k]})
display(pd.DataFrame(config_rows))

expected_logs = [
    'config.txt', 'flow_output', 'rtt_log', 'drop_log', 'link_utilization',
    'buffer_monitor', 'qp_rate_log', 'rate_monitor', 'cnp_log',
    'cnp_trigger_prob_log', 'accumulated_bytes_log', 'pfc_file', 'wan_log', 'log.txt'
]
log_status = []
for name in expected_logs:
    p = Path(ana.dir) / name
    log_status.append({'file': name, 'exists': p.exists(), 'size_bytes': p.stat().st_size if p.exists() else 0})
display(pd.DataFrame(log_status))

## Flow-level summary

这里读取 `flow_output`，展示流数量、intra/inter 分布、慢化比分布和最慢的流。

In [ ]:
flow_df = ana.get_intra_df().copy()
inter_df = ana.get_inter_df().copy()
all_flows = pd.concat([flow_df, inter_df], ignore_index=True)

flow_summary = pd.DataFrame([
    {'scope': 'all', 'flows': len(all_flows), 'avg_slowdown': all_flows['fct_slowdown'].mean(), 'p99_slowdown': all_flows['fct_slowdown'].quantile(0.99), 'avg_size_B': all_flows['fsize'].mean()},
    {'scope': 'intra', 'flows': len(flow_df), 'avg_slowdown': flow_df['fct_slowdown'].mean(), 'p99_slowdown': flow_df['fct_slowdown'].quantile(0.99), 'avg_size_B': flow_df['fsize'].mean()},
    {'scope': 'inter', 'flows': len(inter_df), 'avg_slowdown': inter_df['fct_slowdown'].mean(), 'p99_slowdown': inter_df['fct_slowdown'].quantile(0.99), 'avg_size_B': inter_df['fsize'].mean()},
])
display(flow_summary)

cols = [c for c in ['flow_id', 'src', 'dst', 'src_as', 'dst_as', 'fsize', 'start_time', 'finish_time', 'std_fct', 'fct_slowdown'] if c in all_flows.columns]
display(all_flows.sort_values('fct_slowdown', ascending=False).head(20)[cols])

In [ ]:
if not inter_df.empty:
    pair_summary = (
        inter_df.groupby(['src_as', 'dst_as'])
        .agg(
            flows=('flow_id', 'count'),
            bytes=('fsize', 'sum'),
            avg_slowdown=('fct_slowdown', 'mean'),
            p99_slowdown=('fct_slowdown', lambda s: s.quantile(0.99)),
            max_slowdown=('fct_slowdown', 'max'),
        )
        .reset_index()
        .sort_values(['p99_slowdown', 'avg_slowdown'], ascending=False)
    )
    display(pair_summary.head(20))
else:
    print('No inter-DC flows in this experiment.')

## Built-in diagnostics

这些是 `Analyser` 已有的诊断函数。部分日志不存在时会报错，所以这里统一做了保护。

In [ ]:
def try_call(desc, func, *args, **kwargs):
    print(f'\n## {desc}')
    try:
        ret = func(*args, **kwargs)
        if ret is not None:
            display(ret if not isinstance(ret, tuple) else pd.DataFrame({'value': list(ret)}))
        return ret
    except Exception as e:
        print(f'skipped: {type(e).__name__}: {e}')
        return None

try_call('drop number', ana.get_drop_number)
try_call('drop rate', ana.get_drop_rate)
try_call('WAN buffer stats: mean/p99 MB', ana.get_wan_buffer_stats)

events = try_call(
    'WAN high-buffer intervals, top rows',
    ana.wan_high_buffer_intervals,
    quantile=0.95,
    direction='egress',
    min_duration_s=0,
)
if isinstance(events, pd.DataFrame) and not events.empty:
    display(events.sort_values('peak_bytes', ascending=False).head(20))

In [ ]:
# Slow-flow diagnosis prints a short timeline and displays the slowest flows.
try_call('slow flow diagnosis, threshold=P99', ana.diagnose_slow_flow, 99)
try_call('drop events by switch and AS pair', ana.show_drop)

## Plots

下面这些 cell 直接使用 `deep_analyse.py` 里的画图函数。它们默认在 notebook 中显示；如果某个日志不存在，会跳过。

In [ ]:
try_call('FCT slowdown CDF', ana.plot_fct_cdf)

In [ ]:
# Pick the busiest inter-DC AS pair as the default pair for AS-rate/CNP/RTT/WAN-buffer plots.
if not inter_df.empty:
    top_pair = (
        inter_df.groupby(['src_as', 'dst_as'])['fsize']
        .sum()
        .sort_values(ascending=False)
        .index[0]
    )
    SRC_AS, DST_AS = map(int, top_pair)
    print('default inter-DC pair:', SRC_AS, '->', DST_AS)
else:
    SRC_AS, DST_AS = 0, 1
    print('No inter-DC flows found; using fallback pair 0 -> 1.')

try_call('AS rate monitor', ana.plot_as_rate, SRC_AS, DST_AS)
try_call('RTT monitor', ana.plot_rtt, SRC_AS, DST_AS)
try_call('CNP trigger probability', ana.plot_cnp_trigger_prob, SRC_AS, DST_AS)
try_call('WAN path buffer', ana.plot_wan_path_buffer, SRC_AS, DST_AS, egress=True, unit='MB')

In [ ]:
# Plot queue history for the WAN queue with the highest recorded egress occupancy.
try:
    buf = pd.read_csv(Path(ana.dir) / 'buffer_monitor')
    wan_switches = set(map(int, ana.topo.get('wan_switches', [])))
    wan_buf = buf[buf['switch_id'].isin(wan_switches)] if wan_switches else buf
    if not wan_buf.empty:
        row = wan_buf.sort_values('egress_bytes', ascending=False).iloc[0]
        SWITCH_ID = int(row['switch_id'])
        print('default switch for buffer plot:', SWITCH_ID)
        try_call('buffer monitor for selected switch', ana.plot_buffer, SWITCH_ID, True)
    else:
        print('No buffer samples found.')
except Exception as e:
    print(f'skipped buffer plot: {type(e).__name__}: {e}')

In [ ]:
# Plot QP rate for the slowest inter-DC flows. Adjust FLOW_IDS if you want specific flows.
if not inter_df.empty:
    FLOW_IDS = inter_df.sort_values('fct_slowdown', ascending=False)['flow_id'].head(5).astype(int).tolist()
else:
    FLOW_IDS = all_flows.sort_values('fct_slowdown', ascending=False)['flow_id'].head(5).astype(int).tolist()
print('FLOW_IDS:', FLOW_IDS)
try_call('QP rate for selected slow flows', ana.plot_qp_rate, FLOW_IDS)

In [ ]:
# GSCC W/K helper. Useful when cnp_trigger_prob_log and rate_monitor are available.
try_call('analyze CNP K for weighted GSCC', ana.analyze_cnp_k, w_max=4, start_time=2.01, end_time=2.1)